# Question 2: Optimisation — Telecommunication Tower Placement

**Course:** Artificial Intelligence (ARI711S)  
**Assignment:** 1  
**Group:** Ninjas 2026

---

## Overview

MTC is expanding its 5G network across a **10 × 10 regional grid**. We must place **8 signal boosters** while satisfying strict spatial and environmental constraints.

This is a **Constraint Satisfaction Problem (CSP)** — an extension of the classic N-Queens puzzle into real-world infrastructure planning.

### Problem Formulation

| Element | Description |
|---------|-------------|
| **Variables** | 8 towers: $T_1, T_2, \ldots, T_8$ |
| **Domain** | All 100 cells in the 10×10 grid: $(row, col)$ where $0 \leq row, col \leq 9$ |
| **Initial State** | Empty assignment — no towers placed |
| **Goal State** | All 8 towers placed without violating any constraint |

### Constraints

For any two towers $T_i$ and $T_j$:
1. **Signal Overlap (Row/Column):** No two towers may share the same row or column.
2. **Interference Buffer (Proximity):** No two towers may be placed in adjacent cells, including all 8 diagonal neighbours.
3. **Terrain (Prohibited Zones):** Towers cannot be placed on "Mountain" cells.

### Solution Strategy

We use **backtracking search** with two efficiency improvements:
- **MRV (Minimum Remaining Values):** Always choose the tower whose domain has the fewest valid remaining cells.
- **Forward Checking:** After placing a tower, immediately remove inconsistent values from other towers' domains.

---

## 1. Imports and Setup

In [ ]:
import copy
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

---
## 2. Telecom_CSP_Solver Class

The solver encapsulates:
- The grid size (10×10)
- The number of towers (8)
- Mountain (prohibited) cells
- The domain of each tower (set of valid cells)
- The backtracking algorithm with MRV and Forward Checking

In [ ]:
class Telecom_CSP_Solver:
    """
    Solves the MTC 5G tower placement problem as a CSP.

    Variables  : 8 towers (indexed 0–7)
    Domain     : All valid (non-mountain) cells in a 10×10 grid
    Constraints:
        1. No two towers in the same row or column (signal overlap)
        2. No two towers in adjacent cells, including diagonals (proximity buffer)
        3. No tower on a mountain cell (terrain obstruction)

    Search strategy: Backtracking + MRV heuristic + Forward Checking
    """

    GRID_SIZE    = 10
    NUM_TOWERS   = 8

    def __init__(self, mountains=None):
        """
        Initialise the solver.

        Parameters:
            mountains (list[tuple]): List of (row, col) cells that are
                                     prohibited terrain. Defaults to empty.
        """
        self.mountains = set(mountains) if mountains else set()

        # Build the initial domain: every non-mountain cell is valid
        all_cells = [
            (r, c)
            for r in range(self.GRID_SIZE)
            for c in range(self.GRID_SIZE)
        ]
        initial_domain = [
            cell for cell in all_cells if cell not in self.mountains
        ]

        # domains[i] = set of cells still available for tower i
        self.domains = [set(initial_domain) for _ in range(self.NUM_TOWERS)]

        self.solution   = None    # Stores a valid assignment once found
        self.node_count = 0       # Counts recursive calls (search effort)

        print(f"Grid       : {self.GRID_SIZE} x {self.GRID_SIZE}")
        print(f"Towers     : {self.NUM_TOWERS}")
        print(f"Mountains  : {sorted(self.mountains)}")
        print(f"Initial domain size per tower: {len(initial_domain)} cells")

    # ------------------------------------------------------------------
    # Constraint: is_consistent
    # ------------------------------------------------------------------
    def is_consistent(self, assignment, cell):
        """
        Check whether placing a new tower at 'cell' is consistent with all
        currently assigned towers.

        Violated if ANY already-placed tower:
          - shares the same row or column  (signal overlap)
          - is within 1 step in any direction, including diagonals (interference)

        Parameters:
            assignment (dict): {tower_index: (row, col)} already placed
            cell       (tuple): (row, col) candidate position

        Returns:
            bool: True if consistent, False if any constraint is violated.
        """
        r_new, c_new = cell

        for _, (r_placed, c_placed) in assignment.items():
            # Constraint 1: same row or same column
            if r_new == r_placed or c_new == c_placed:
                return False

            # Constraint 2: adjacent cell (including diagonals)
            # Chebyshev distance == 1 means touching in any direction
            if abs(r_new - r_placed) <= 1 and abs(c_new - c_placed) <= 1:
                return False

        return True

    # ------------------------------------------------------------------
    # MRV: select the unassigned variable with the smallest domain
    # ------------------------------------------------------------------
    def select_unassigned_variable(self, assignment, domains):
        """
        Minimum Remaining Values (MRV) heuristic.

        Among all unassigned towers, choose the one whose domain currently
        contains the fewest valid cells. This focuses search on the most
        constrained variable first, pruning failures early.

        Parameters:
            assignment (dict)        : {tower_index: cell} current placement
            domains    (list[set])   : Current domain for each tower

        Returns:
            int: Index of the tower to assign next.
        """
        unassigned = [
            i for i in range(self.NUM_TOWERS) if i not in assignment
        ]
        # Pick the tower with the minimum domain size
        return min(unassigned, key=lambda i: len(domains[i]))

    # ------------------------------------------------------------------
    # Forward Checking: prune domains after a placement
    # ------------------------------------------------------------------
    def forward_check(self, assignment, tower_idx, cell, domains):
        """
        Forward Checking: after placing tower_idx at cell, remove all cells
        from unassigned towers' domains that would violate a constraint.

        Parameters:
            assignment (dict)      : Current placement
            tower_idx  (int)       : Index of the tower just placed
            cell       (tuple)     : (row, col) where tower_idx was placed
            domains    (list[set]) : Domains to prune (will be modified)

        Returns:
            bool: False if any domain becomes empty (dead end), True otherwise.
        """
        r_placed, c_placed = cell

        for other in range(self.NUM_TOWERS):
            if other in assignment:
                continue  # Already placed; skip

            # Find cells in the other tower's domain that are now invalid
            to_remove = set()
            for candidate in domains[other]:
                r_c, c_c = candidate
                # Same row or column → signal overlap
                if r_c == r_placed or c_c == c_placed:
                    to_remove.add(candidate)
                    continue
                # Adjacent (including diagonal) → interference buffer
                if abs(r_c - r_placed) <= 1 and abs(c_c - c_placed) <= 1:
                    to_remove.add(candidate)

            domains[other] -= to_remove

            # If the domain is now empty, this branch is a dead end
            if not domains[other]:
                return False

        return True

    # ------------------------------------------------------------------
    # Backtracking Search
    # ------------------------------------------------------------------
    def backtrack(self, assignment, domains):
        """
        Core recursive backtracking function.

        At each step:
          1. Select the next tower using MRV.
          2. Try each cell in its domain.
          3. Check consistency before placement.
          4. Apply forward checking to prune other domains.
          5. Recurse; backtrack if no solution found.

        Parameters:
            assignment (dict)      : {tower_index: (row, col)} current state
            domains    (list[set]) : Current domain for each tower

        Returns:
            dict or None: Complete valid assignment, or None if none exists.
        """
        self.node_count += 1

        # Base case: all towers are placed
        if len(assignment) == self.NUM_TOWERS:
            return assignment

        # MRV: choose most-constrained unassigned tower
        tower = self.select_unassigned_variable(assignment, domains)

        # Order the domain cells — sorted for deterministic output
        for cell in sorted(domains[tower]):
            # Check whether this cell is consistent with current placement
            if self.is_consistent(assignment, cell):
                # Place the tower
                assignment[tower] = cell

                # Deep-copy domains so we can restore on backtrack
                new_domains = copy.deepcopy(domains)
                new_domains[tower] = {cell}  # Fix this tower's domain

                # Forward checking: prune other domains
                if self.forward_check(assignment, tower, cell, new_domains):
                    result = self.backtrack(assignment, new_domains)
                    if result is not None:
                        return result  # Solution found

                # Backtrack: remove placement and restore
                del assignment[tower]

        return None  # No valid placement found from this state

    # ------------------------------------------------------------------
    # Public: solve
    # ------------------------------------------------------------------
    def solve(self):
        """
        Initiate the backtracking search from an empty assignment.

        Returns:
            dict or None: {tower_index: (row, col)} if solved, else None.
        """
        print("\nSearching for a valid tower placement...")
        self.node_count = 0
        result = self.backtrack({}, copy.deepcopy(self.domains))

        if result is not None:
            self.solution = result
            print(f"Solution found after {self.node_count} recursive calls.")
            print("\nTower placements:")
            for t_idx in sorted(result):
                r, c = result[t_idx]
                print(f"  T{t_idx + 1}: row={r}, col={c}")
        else:
            print("No solution found.")

        return result

    # ------------------------------------------------------------------
    # Visualise
    # ------------------------------------------------------------------
    def visualise(self, assignment, title="MTC Tower Placement", save_as=None):
        """
        Plot the 10×10 grid showing tower and mountain positions.

        Colour scheme:
            Light green — open cell
            Brown       — mountain (prohibited zone)
            Steel blue  — tower cell

        Parameters:
            assignment (dict): {tower_index: (row, col)}
            title      (str) : Plot title
            save_as    (str) : Filename to save (optional)
        """
        if assignment is None:
            print("No assignment to visualise.")
            return

        tower_cells = set(assignment.values())

        COLOR_OPEN     = [0.94, 0.97, 0.94]   # very light green
        COLOR_MOUNTAIN = [0.60, 0.40, 0.20]   # brown
        COLOR_TOWER    = [0.18, 0.45, 0.71]   # steel blue

        image = np.ones((self.GRID_SIZE, self.GRID_SIZE, 3))
        for r in range(self.GRID_SIZE):
            for c in range(self.GRID_SIZE):
                if (r, c) in self.mountains:
                    image[r, c] = COLOR_MOUNTAIN
                elif (r, c) in tower_cells:
                    image[r, c] = COLOR_TOWER
                else:
                    image[r, c] = COLOR_OPEN

        fig, ax = plt.subplots(figsize=(7, 7))
        ax.imshow(image, interpolation='nearest')

        # Draw grid lines
        ax.set_xticks(np.arange(-0.5, self.GRID_SIZE, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, self.GRID_SIZE, 1), minor=True)
        ax.grid(which='minor', color='#888888', linewidth=0.8)
        ax.set_xticks(range(self.GRID_SIZE))
        ax.set_yticks(range(self.GRID_SIZE))
        ax.set_xticklabels(range(self.GRID_SIZE), fontsize=8)
        ax.set_yticklabels(range(self.GRID_SIZE), fontsize=8)
        ax.tick_params(which='minor', length=0)
        ax.set_xlabel("Column", fontsize=10)
        ax.set_ylabel("Row", fontsize=10)

        # Label towers
        tower_idx_map = {v: k for k, v in assignment.items()}
        for r in range(self.GRID_SIZE):
            for c in range(self.GRID_SIZE):
                if (r, c) in tower_cells:
                    t_num = tower_idx_map[(r, c)] + 1
                    ax.text(c, r, f"T{t_num}",
                            ha='center', va='center',
                            fontsize=9, fontweight='bold', color='white')
                elif (r, c) in self.mountains:
                    ax.text(c, r, 'M',
                            ha='center', va='center',
                            fontsize=9, fontweight='bold', color='white')

        # Legend
        legend_items = [
            mpatches.Patch(color=COLOR_TOWER,    label='Signal Booster (T)'),
            mpatches.Patch(color=COLOR_MOUNTAIN, label='Mountain / Prohibited (M)'),
            mpatches.Patch(color=COLOR_OPEN,     label='Open Cell'),
        ]
        ax.legend(handles=legend_items, loc='upper right',
                  fontsize=8, framealpha=0.9, bbox_to_anchor=(1.38, 1.0))

        ax.set_title(title, fontsize=12, pad=12)
        plt.tight_layout()

        if save_as:
            plt.savefig(save_as, dpi=150, bbox_inches='tight')
            print(f"Plot saved as '{save_as}'")
        plt.show()

    # ------------------------------------------------------------------
    # Verify solution
    # ------------------------------------------------------------------
    def verify_solution(self, assignment):
        """
        Independently verify that the solution satisfies all constraints.

        Returns:
            bool: True if all constraints are satisfied.
        """
        if assignment is None:
            print("Nothing to verify.")
            return False

        cells = list(assignment.values())
        passed = True

        # Check mountain constraint
        for idx, cell in assignment.items():
            if cell in self.mountains:
                print(f"  FAIL: T{idx+1} at {cell} is on a mountain.")
                passed = False

        # Check pairwise constraints
        for i in range(len(cells)):
            for j in range(i + 1, len(cells)):
                r1, c1 = cells[i]
                r2, c2 = cells[j]
                t1 = list(assignment.keys())[i] + 1
                t2 = list(assignment.keys())[j] + 1

                if r1 == r2:
                    print(f"  FAIL: T{t1} and T{t2} share row {r1}.")
                    passed = False
                if c1 == c2:
                    print(f"  FAIL: T{t1} and T{t2} share column {c1}.")
                    passed = False
                if abs(r1 - r2) <= 1 and abs(c1 - c2) <= 1:
                    print(f"  FAIL: T{t1}{cells[i]} and T{t2}{cells[j]} are adjacent.")
                    passed = False

        if passed:
            print("  All constraints satisfied. Solution is VALID.")
        return passed

---
## 3. Test Scenario — Level 1: Coastal Region (Easy)

**Mountains:** `[(0,0), (1,1), (9,9)]`  
**Objective:** Establish basic row, column, and diagonal proximity logic with minimal terrain obstruction.

In [ ]:
print("=" * 55)
print("LEVEL 1 — COASTAL REGION (EASY)")
print("=" * 55)

mountains_lvl1 = [(0, 0), (1, 1), (9, 9)]

solver1 = Telecom_CSP_Solver(mountains=mountains_lvl1)
solution1 = solver1.solve()

print("\nVerification:")
solver1.verify_solution(solution1)

In [ ]:
solver1.visualise(
    assignment=solution1,
    title="Level 1 — Coastal Region (Easy)\nMountains: (0,0), (1,1), (9,9)",
    save_as="towers_level1_coastal.png"
)

---
## 4. Test Scenario — Level 2: Highlands Region (Medium)

**Mountains:** `[(2,2), (2,3), (3,2), (3,3), (7,8), (8,7), (8,8)]`  
**Objective:** Navigate around clusters of prohibited terrain — a 2×2 block in the upper-centre and a 2×2 cluster in the lower-right.

In [ ]:
print("=" * 55)
print("LEVEL 2 — HIGHLANDS REGION (MEDIUM)")
print("=" * 55)

mountains_lvl2 = [(2, 2), (2, 3), (3, 2), (3, 3), (7, 8), (8, 7), (8, 8)]

solver2 = Telecom_CSP_Solver(mountains=mountains_lvl2)
solution2 = solver2.solve()

print("\nVerification:")
solver2.verify_solution(solution2)

In [ ]:
solver2.visualise(
    assignment=solution2,
    title="Level 2 — Highlands Region (Medium)\nMountain clusters at (2-3, 2-3) and (7-8, 7-8)",
    save_as="towers_level2_highlands.png"
)

---
## 5. Test Scenario — Level 3: Brandberg Region (Hard)

**Mountains:** `[(0,5), (1,5), (2,5), (3,5), (4,5), (5,0), (5,1), (5,2), (5,3), (5,4)]`  
**Objective:** Mountains form a partial **"L-shaped wall"**, heavily restricting available positions and forcing towers into tight constraint clusters.

In [ ]:
print("=" * 55)
print("LEVEL 3 — BRANDBERG REGION (HARD)")
print("=" * 55)

mountains_lvl3 = [
    (0, 5), (1, 5), (2, 5), (3, 5), (4, 5),
    (5, 0), (5, 1), (5, 2), (5, 3), (5, 4)
]

solver3 = Telecom_CSP_Solver(mountains=mountains_lvl3)
solution3 = solver3.solve()

print("\nVerification:")
solver3.verify_solution(solution3)

In [ ]:
solver3.visualise(
    assignment=solution3,
    title="Level 3 — Brandberg Region (Hard)\nL-shaped mountain wall blocking column 5 and row 5",
    save_as="towers_level3_brandberg.png"
)

---
## 6. Comparative Summary Across All Levels

In [ ]:
# Side-by-side comparison of all three solutions
scenarios = [
    ("Level 1\nCoastal (Easy)",   solver1, solution1),
    ("Level 2\nHighlands (Medium)", solver2, solution2),
    ("Level 3\nBrandberg (Hard)",  solver3, solution3),
]

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

COLOR_OPEN     = [0.94, 0.97, 0.94]
COLOR_MOUNTAIN = [0.60, 0.40, 0.20]
COLOR_TOWER    = [0.18, 0.45, 0.71]

for ax, (label, solver, assignment) in zip(axes, scenarios):
    if assignment is None:
        ax.set_title(f"{label}\nNO SOLUTION")
        continue

    tower_cells = set(assignment.values())
    tower_idx_map = {v: k for k, v in assignment.items()}

    image = np.ones((solver.GRID_SIZE, solver.GRID_SIZE, 3))
    for r in range(solver.GRID_SIZE):
        for c in range(solver.GRID_SIZE):
            if (r, c) in solver.mountains:
                image[r, c] = COLOR_MOUNTAIN
            elif (r, c) in tower_cells:
                image[r, c] = COLOR_TOWER
            else:
                image[r, c] = COLOR_OPEN

    ax.imshow(image, interpolation='nearest')

    ax.set_xticks(np.arange(-0.5, solver.GRID_SIZE, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, solver.GRID_SIZE, 1), minor=True)
    ax.grid(which='minor', color='#888888', linewidth=0.6)
    ax.set_xticks(range(solver.GRID_SIZE))
    ax.set_yticks(range(solver.GRID_SIZE))
    ax.set_xticklabels(range(solver.GRID_SIZE), fontsize=7)
    ax.set_yticklabels(range(solver.GRID_SIZE), fontsize=7)
    ax.tick_params(which='minor', length=0)

    for r in range(solver.GRID_SIZE):
        for c in range(solver.GRID_SIZE):
            if (r, c) in tower_cells:
                t_num = tower_idx_map[(r, c)] + 1
                ax.text(c, r, f"T{t_num}",
                        ha='center', va='center',
                        fontsize=8, fontweight='bold', color='white')
            elif (r, c) in solver.mountains:
                ax.text(c, r, 'M',
                        ha='center', va='center',
                        fontsize=8, fontweight='bold', color='white')

    ax.set_title(
        f"{label}\n"
        f"Nodes explored: {solver.node_count}  |  Mountains: {len(solver.mountains)}",
        fontsize=10
    )

legend_items = [
    mpatches.Patch(color=COLOR_TOWER,    label='Signal Booster (T)'),
    mpatches.Patch(color=COLOR_MOUNTAIN, label='Mountain (M)'),
    mpatches.Patch(color=COLOR_OPEN,     label='Open Cell'),
]
fig.legend(handles=legend_items, loc='lower center',
           ncol=3, fontsize=9, framealpha=0.9)

plt.suptitle("MTC 5G Tower Placement — All Scenarios", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("towers_all_levels.png", dpi=150, bbox_inches='tight')
plt.show()
print("Comparison image saved as 'towers_all_levels.png'")

---
## 7. Summary and Analysis

### Algorithm Summary

| Component | Description |
|-----------|-------------|
| **Backtracking** | Core recursive DFS that places one tower per call, undoing choices that lead to dead ends |
| **MRV Heuristic** | Selects the tower with the fewest remaining valid cells next — fails fast on tight constraints |
| **Forward Checking** | After each placement, removes newly invalid cells from all unplaced towers' domains |
| **is_consistent** | Validates row/column separation and proximity (Chebyshev distance > 1) before each placement |

### Constraint Interactions

The three constraints together are significantly stricter than N-Queens alone:
- **Row/column exclusion** eliminates at least 19 cells per placed tower (1 row + 1 column)
- **Proximity buffer (radius 1)** eliminates up to 8 additional adjacent cells
- **Mountain terrain** further reduces the available domain

### Level Comparison

| Level | Mountains | Difficulty | Reason |
|-------|-----------|------------|--------|
| 1 — Coastal | 3 (scattered) | Easy | Mountains in corners/isolated — little interference |
| 2 — Highlands | 7 (clustered) | Medium | 2×2 blocks eliminate entire zones; cascades through MRV |
| 3 — Brandberg | 10 (L-shaped wall) | Hard | Partial wall splits the grid; forces towers into the most constrained half |

### Why MRV + Forward Checking Works Well

Without MRV, a naive backtracker might spend thousands of iterations on easy towers while the hard ones are left until late — only to fail then. MRV front-loads the hardest decisions, and forward checking ensures impossible branches are detected **before** recursing into them, dramatically reducing the search space.

### Admissibility Note (Heuristic)

MRV is not strictly a heuristic in the pathfinding sense — it is a **variable ordering strategy**. It is complete (will find a solution if one exists) and sound (never incorrectly prunes valid solutions), making it a safe and effective addition to backtracking.